# Test Samples — Track A vs Track B Comparison

This notebook smoke-tests both pipelines on the three Dutch samples in `audio/test_sample/` to help decide which transcriber + diarizer combination to use for the 2 h meeting.

**Note:** This notebook makes real API calls to AssemblyAI, ElevenLabs, and pyannoteAI. Ensure you have the required API keys in your `.env` file.

## Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime, timezone
import json
import subprocess

load_dotenv()

from meetings.config import get_settings
from meetings.audio import audio_meta
from meetings.pipelines.assemblyai import AssemblyAIPipeline
from meetings.pipelines.custom import CustomPipeline
from meetings.notebook_helpers import compute_gross_metrics
from meetings.io import read_transcript

settings = get_settings()

# List the three sample paths
TEST_SAMPLES = [
    Path("audio/test_sample/segment_1_start.wav"),
    Path("audio/test_sample/segment_2_45min.wav"),
    Path("audio/test_sample/segment_3_90min.wav"),
]

print("Test samples:")
for sample in TEST_SAMPLES:
    print(f"  {sample}")

In [ ]:
# Verify all samples are 16 kHz mono PCM
def check_audio_format(path: Path) -> dict:
    """Check if audio is 16 kHz mono PCM using ffprobe."""
    result = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "stream=codec_name,sample_rate,channels",
            "-of",
            "json",
            str(path),
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        return {"error": result.stderr}
    
    data = json.loads(result.stdout)
    if not data.get("streams"):
        return {"error": "No streams found"}
    
    stream = data["streams"][0]
    return {
        "codec": stream.get("codec_name"),
        "sample_rate": int(stream.get("sample_rate", 0)),
        "channels": stream.get("channels"),
    }

print("Checking audio formats:")
all_valid = True
for sample in TEST_SAMPLES:
    if not sample.exists():
        print(f"  ❌ {sample.name}: file not found")
        all_valid = False
        continue
    
    fmt = check_audio_format(sample)
    if "error" in fmt:
        print(f"  ❌ {sample.name}: {fmt['error']}")
        all_valid = False
        continue
    
    is_valid = (
        fmt["codec"] == "pcm_s16le"
        and fmt["sample_rate"] == 16000
        and fmt["channels"] == 1
    )
    
    if is_valid:
        print(f"  ✓ {sample.name}: {fmt['codec']}, {fmt['sample_rate']} Hz, {fmt['channels']} ch")
    else:
        print(f"  ❌ {sample.name}: {fmt['codec']}, {fmt['sample_rate']} Hz, {fmt['channels']} ch (expected pcm_s16le, 16000 Hz, 1 ch)")
        all_valid = False

if not all_valid:
    print("\n⚠️  Some samples are not in 16 kHz mono PCM format.")
    print("   Run: uv run meetings preprocess <audio_file>")
else:
    print("\n✓ All samples are in the correct format.")

## Track A — AssemblyAI

Run AssemblyAI transcription + diarization on all three samples.

In [ ]:
if not settings.assemblyai_api_key:
    print("⚠️  ASSEMBLYAI_API_KEY not set. Skipping Track A.")
    aai_results = {}
else:
    print("Running Track A (AssemblyAI) on all samples...")
    aai_pipeline = AssemblyAIPipeline()
    aai_results = {}
    
    for sample in TEST_SAMPLES:
        if not sample.exists():
            print(f"  Skipping {sample.name}: file not found")
            continue
        
        utc_now = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        run_dir = settings.transcription_dir / "_test" / f"aai__{sample.stem}__{utc_now}"
        
        print(f"  Processing {sample.name} -> {run_dir.name}...")
        try:
            transcript = aai_pipeline.transcribe(sample, run_dir, language="nl")
            aai_results[sample.stem] = {"run_dir": run_dir, "transcript": transcript}
            print(f"    ✓ Done: {len(transcript.speakers)} speakers, {len(transcript.segments)} segments")
        except Exception as e:
            print(f"    ❌ Failed: {e}")
            aai_results[sample.stem] = {"error": str(e)}
    
    print("\nTrack A complete.")

## Track B — Scribe v2 + pyannoteAI

Run CustomPipeline with ElevenLabs Scribe v2 transcription and pyannoteAI diarization on all three samples.

In [ ]:
if not settings.elevenlabs_api_key or not settings.pyannoteai_api_key:
    print("⚠️  ELEVENLABS_API_KEY or PYANNOTEAI_API_KEY not set. Skipping Track B.")
    custom_results = {}
else:
    print("Running Track B (Scribe v2 + pyannoteAI) on all samples...")
    custom_pipeline = CustomPipeline(
        transcriber="elevenlabs",
        diarizer="pyannoteai",
        summarizer="claude",  # Not used in transcribe_only
    )
    custom_results = {}
    
    for sample in TEST_SAMPLES:
        if not sample.exists():
            print(f"  Skipping {sample.name}: file not found")
            continue
        
        utc_now = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        run_dir = settings.transcription_dir / "_test" / f"custom__{sample.stem}__{utc_now}"
        
        print(f"  Processing {sample.name} -> {run_dir.name}...")
        try:
            transcript = custom_pipeline.transcribe_only(sample, run_dir, language="nl")
            custom_results[sample.stem] = {"run_dir": run_dir, "transcript": transcript}
            print(f"    ✓ Done: {len(transcript.speakers)} speakers, {len(transcript.segments)} segments")
        except Exception as e:
            print(f"    ❌ Failed: {e}")
            custom_results[sample.stem] = {"error": str(e)}
    
    print("\nTrack B complete.")

## Side-by-Side Comparison

For each sample, display metrics and the first 30 seconds of each transcript.

In [ ]:
def format_timestamp(seconds: float) -> str:
    """Format seconds as MM:SS."""
    mins = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{mins:02d}:{secs:02d}"

def render_first_30s(transcript) -> str:
    """Render the first 30 seconds of transcript with speaker labels."""
    lines = []
    cutoff = 30.0
    
    for seg in transcript.segments:
        if seg.start >= cutoff:
            break
        
        speaker = seg.speaker or "UNKNOWN"
        time_str = format_timestamp(seg.start)
        lines.append(f"[{time_str}] {speaker}: {seg.text}")
    
    return "\n".join(lines)

for sample in TEST_SAMPLES:
    stem = sample.stem
    print(f"\n{'='*80}")
    print(f"Sample: {stem}")
    print(f"{'='*80}")
    
    # Track A
    print(f"\n--- Track A: AssemblyAI ---")
    if stem in aai_results:
        result = aai_results[stem]
        if "error" in result:
            print(f"Error: {result['error']}")
        else:
            transcript = result["transcript"]
            metrics = compute_gross_metrics(transcript)
            print(f"Speakers: {metrics['n_speakers']}")
            print(f"Segments: {metrics['n_segments']}")
            print(f"Words: {metrics['n_words']}")
            print(f"Duration: {metrics['duration']:.1f}s")
            print(f"\nFirst 30 seconds:")
            print(render_first_30s(transcript))
    else:
        print("Not run (missing API key or skipped)")
    
    # Track B
    print(f"\n--- Track B: Scribe v2 + pyannoteAI ---")
    if stem in custom_results:
        result = custom_results[stem]
        if "error" in result:
            print(f"Error: {result['error']}")
        else:
            transcript = result["transcript"]
            metrics = compute_gross_metrics(transcript)
            print(f"Speakers: {metrics['n_speakers']}")
            print(f"Segments: {metrics['n_segments']}")
            print(f"Words: {metrics['n_words']}")
            print(f"Duration: {metrics['duration']:.1f}s")
            print(f"\nFirst 30 seconds:")
            print(render_first_30s(transcript))
    else:
        print("Not run (missing API key or skipped)")

## Decision

Based on the side-by-side comparison above, decide which track and combination to use for the 2 h meeting.

### Decision Template

Fill in your choice below:

```
Selected Track: [Track A / Track B]

If Track A:
  Backend: assemblyai (Universal-2)

If Track B:
  Transcriber: [elevenlabs / openai / deepgram]
  Diarizer: [pyannoteai / pyannote_local]
  Summarizer: [claude / gemini]

Rationale:
- [Quality/Accuracy considerations]
- [Speed/Cost considerations]
- [Other factors]
```

### CLI Command for 2 h Run

Once you've decided, run the full 2 h meeting from the terminal:

```powershell
# Track A
uv run meetings transcribe --backend assemblyai --audio "audio/processed/<meeting>.16k.mono.wav" --language nl

# Track B (example with Scribe v2 + pyannoteAI)
uv run meetings transcribe --backend custom --transcriber elevenlabs --diarizer pyannoteai --audio "audio/processed/<meeting>.16k.mono.wav" --language nl
```